# Reinforcement Learning Approach

In [1]:
# load data
import pandas as pd

df = pd.read_csv("wide_processed_final.csv")
df.head()

,word1,word2,word3,word4,category,difficulty,General Category
0,curses,fudge,blast,crud,"""aw, heck!""",medium,Idioms & Slang
1,choral,jazz,rap,americana,"""best ___ performance"" grammy award",very_hard,Fill-in-the-Blank
2,lord,please,sheesh,brother,"""give me a break!""",easy,Idioms & Slang
3,heavens,gracious,mercy,dear,"""my goodness!""",easy,Idioms & Slang
4,piece of cake,no sweat,easy,child’s play,"""nothing to it!""",easy,Idioms & Slang


In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import pickle

print("Loading dataset...")
df = pd.read_csv("wide_processed_final.csv")

# Extract vocabulary
words = pd.concat([df['word1'], df['word2'], df['word3'], df['word4']])
words = words.astype(str).str.strip().str.lower().unique().tolist()
print(f"Found {len(words)} unique words.")

# Load the fast transformer
print("Loading SentenceTransformer ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode words
print("Encoding... (Takes about 1-2 minutes)")
embeddings = model.encode(words, show_progress_bar=True, convert_to_numpy=True)

word_to_vec = {word: embeddings[i] for i, word in enumerate(words)}

# Save to pickle
with open("word_embeddings_dict.pkl", 'wb') as f:
    pickle.dump(word_to_vec, f)
print("Saved word_embeddings_dict.pkl successfully!")

Loading dataset...
Found 6886 unique words.
Loading SentenceTransformer ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding... (Takes about 1-2 minutes)


Batches:   0%|          | 0/216 [00:00<?, ?it/s]

Saved word_embeddings_dict.pkl successfully!


In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import random
import pickle

class ConnectionsEnv(gym.Env):
    def __init__(self, csv_path="wide_processed_final.csv", dict_path="word_embeddings_dict.pkl"):
        super(ConnectionsEnv, self).__init__()
        
        self.df = pd.read_csv(csv_path)
        with open(dict_path, 'rb') as f:
            self.word_to_vec = pickle.load(f)
            
        # Action space: Pick 4 indices from the 16 slots
        self.action_space = spaces.MultiDiscrete([16, 16, 16, 16])
        
        # Observation space: 16 vectors of size 384, plus 1 discrete value for lives
        self.observation_space = spaces.Dict({
            "board_vectors": spaces.Box(low=-10.0, high=10.0, shape=(16, 384), dtype=np.float32),
            "lives_remaining": spaces.Discrete(5)
        })
        
        self.current_board = []
        self.solution_groups = []
        self.remaining_indices = []
        self.lives = 4
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Keep sampling until we find a perfect 16-word combination without duplicates
        while True:
            sampled_rows = self.df.sample(4)
            self.solution_groups = []
            words_pool = []
            valid_board = True
            
            for _, row in sampled_rows.iterrows():
                group = {str(row['word1']).strip().lower(), 
                         str(row['word2']).strip().lower(), 
                         str(row['word3']).strip().lower(), 
                         str(row['word4']).strip().lower()}
                
                if len(group) != 4:
                    valid_board = False
                    break
                    
                self.solution_groups.append(group)
                words_pool.extend(list(group))
                
            if valid_board and len(set(words_pool)) == 16:
                break
                
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        
        return self._get_obs(), {}
        
    def _get_obs(self):
        # Create a blank array of zeroes
        vectors = np.zeros((16, 384), dtype=np.float32)
        
        # Fill in the vectors for words that are still in play
        for i, word in enumerate(self.current_board):
            if i in self.remaining_indices:
                vectors[i] = self.word_to_vec.get(word, np.zeros(384))
                
        return {
            "board_vectors": vectors,
            "lives_remaining": self.lives
        }

    def step(self, action):
        guess_indices = set(action)
        if len(guess_indices) != 4 or not guess_indices.issubset(set(self.remaining_indices)):
            return self._get_obs(), -5, False, False, {"info": "Invalid action"}
            
        guessed_words = {self.current_board[i] for i in guess_indices}
        
        reward = 0
        terminated = False
        info = {}
        
        max_overlap = 0
        matched_group = None
        
        for group in self.solution_groups:
            overlap = len(guessed_words.intersection(group))
            if overlap > max_overlap:
                max_overlap = overlap
                matched_group = group

        if max_overlap == 4:
            reward = 10
            self.remaining_indices = [i for i in self.remaining_indices if i not in guess_indices]
            self.solution_groups.remove(matched_group)
            info["status"] = "Correct!"
            if len(self.remaining_indices) == 0:
                reward += 20
                terminated = True
                info["status"] = "Game Won!"
        elif max_overlap == 3:
            reward = 2
            self.lives -= 1
            info["status"] = "One Away!"
        else:
            reward = -1
            self.lives -= 1
            info["status"] = "Incorrect."
            
        if self.lives <= 0 and not terminated:
            reward = -10
            terminated = True
            info["status"] = "Game Over"
            
        return self._get_obs(), reward, terminated, False, info

In [4]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class ConnectionsFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super(ConnectionsFeatureExtractor, self).__init__(observation_space, features_dim)
        
        # 16 words * 384 dimensions + 5 lives = 6149 inputs
        input_size = (16 * 384) + 5 
        
        self.linear = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, features_dim),
            nn.ReLU()
        )

    def forward(self, observations):
        # Shape: (BatchSize, 16, 384)
        board_vectors = observations["board_vectors"]  
        batch_size = board_vectors.shape[0]
        
        # Flatten the board vectors to (BatchSize, 6144)
        flattened_board = board_vectors.reshape(batch_size, -1)
        
        # Safely force lives to (BatchSize, 5)
        lives = observations["lives_remaining"].float().view(batch_size, -1)
        
        # Concatenate into (BatchSize, 6149)
        rl_state = torch.cat([flattened_board, lives], dim=1)
        
        return self.linear(rl_state)

In [5]:
from stable_baselines3.common.callbacks import BaseCallback
from IPython.display import clear_output

class CleanOutputCallback(BaseCallback):
    """
    Custom callback that intercepts the training loop and clears 
    the Jupyter output right before printing the newest log table.
    """
    def _on_rollout_end(self) -> bool:
        # wait=True prevents screen flickering by waiting for the 
        # new text to be ready before deleting the old text
        clear_output(wait=True)
        return True

    def _on_step(self) -> bool:
        # We don't need to do anything on individual steps
        return True

In [14]:
from stable_baselines3 import PPO

print("Initializing Environment...")
env = ConnectionsEnv()

policy_kwargs = dict(
    features_extractor_class=ConnectionsFeatureExtractor,
    features_extractor_kwargs=dict(features_dim=256),
)

print("Initializing PPO Agent Phase 2...")
agent = PPO(
    "MultiInputPolicy", 
    env, 
    policy_kwargs=policy_kwargs, 
    learning_rate=0.00005,      # <-- LOWERED: Stops the KL Divergence thrashing
    ent_coef=0.05,              # <-- ADDED: Forces the agent to keep exploring, preventing "confident losing"
    batch_size=128,             # <-- ADDED: Smaller batches help it learn from rare wins (One Aways) faster
    verbose=1,
    tensorboard_log="./connections_tensorboard/"
)

# Optional: If you want to continue training the EXACT SAME agent instead of starting over:
# agent = PPO.load("nyt_connections_rl_agent", env=env)
# agent.learning_rate = 0.00005
# agent.ent_coef = 0.05

print("Starting Deep Training (500,000 steps)...")
agent.learn(total_timesteps=500_000, tb_log_name="Deep_Run_1", callback=CleanOutputCallback())

print("Saving Agent...")
agent.save("nyt_connections_rl_agent_v2")
print("Phase 2 Training Complete!")

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 5.67         |
|    ep_rew_mean          | -20.4        |
| time/                   |              |
|    fps                  | 275          |
|    iterations           | 27           |
|    time_elapsed         | 200          |
|    total_timesteps      | 55296        |
| train/                  |              |
|    approx_kl            | 0.0069273715 |
|    clip_fraction        | 0.0307       |
|    clip_range           | 0.2          |
|    entropy_loss         | -10.9        |
|    explained_variance   | 6.27e-05     |
|    learning_rate        | 5e-05        |
|    loss                 | 69.3         |
|    n_updates            | 260          |
|    policy_gradient_loss | -0.00764     |
|    value_loss           | 102          |
------------------------------------------


KeyboardInterrupt: 

In [16]:
# run "tensorboard --logdir ./connections_tensorboard/ --port 6006" for graphs

# looking at ep_rew_mean to increase to ideally 60

# entropy loss should ideally converging near zero

# ep_len_mean should be nearing 5-6, this goes in hand with score

# approx_kl should be a stable bounce near 0.01, big jumps to 0.1 might indicate LR is too high